# 01 — Bronze Ingestion: Connected Vehicle Telemetry

This notebook generates synthetic vehicle telemetry data and writes it as raw Delta tables
into the **Bronze** lakehouse. Data includes:
- Vehicle sensor readings (engine temp, oil pressure, tire pressure, battery voltage, RPM)
- Diagnostic Trouble Codes (DTCs)
- Maintenance event records

**Medallion Layer:** Bronze (raw, append-only)

In [ ]:
# Bronze layer Spark config: optimized for write-heavy ingestion
spark.conf.set("spark.sql.parquet.vorder.enabled", "false")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.merge.lowShuffle.enabled", "true")

In [ ]:
import random
import uuid
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, IntegerType, ArrayType
)
from pyspark.sql.functions import current_timestamp, lit

## Generate Vehicle Fleet Master Data

In [ ]:
# Fleet configuration
NUM_VEHICLES = 100
MAKES_MODELS = [
    ("Toyota", "Camry", 2020), ("Toyota", "RAV4", 2021), ("Toyota", "Corolla", 2022),
    ("Honda", "Civic", 2021), ("Honda", "CR-V", 2020), ("Honda", "Accord", 2023),
    ("Ford", "F-150", 2022), ("Ford", "Explorer", 2021), ("Ford", "Mustang", 2023),
    ("Chevrolet", "Silverado", 2022), ("Chevrolet", "Equinox", 2021),
    ("BMW", "3 Series", 2023), ("BMW", "X5", 2022),
    ("Tesla", "Model 3", 2023), ("Tesla", "Model Y", 2022),
    ("Hyundai", "Tucson", 2023), ("Hyundai", "Elantra", 2022),
    ("Kia", "Sportage", 2023), ("Kia", "Forte", 2022),
    ("Nissan", "Altima", 2021), ("Nissan", "Rogue", 2022)
]
REGIONS = ["Northeast", "Southeast", "Midwest", "Southwest", "West", "Pacific NW"]

random.seed(42)
vehicles = []
for i in range(NUM_VEHICLES):
    make, model, year = random.choice(MAKES_MODELS)
    vin = f"1{''.join(random.choices('ABCDEFGHJKLMNPRSTUVWXYZ0123456789', k=16))}"
    vehicles.append(Row(
        vin=vin,
        make=make,
        model=model,
        year=year,
        mileage=random.randint(5000, 120000),
        region=random.choice(REGIONS),
        fleet_id=f"FLEET-{random.randint(1, 10):03d}"
    ))

df_vehicles = spark.createDataFrame(vehicles)
df_vehicles.write.format("delta").mode("overwrite").save("Tables/raw_vehicles")
print(f"Wrote {df_vehicles.count()} vehicles to Bronze raw_vehicles")

## Generate Sensor Telemetry Data

In [ ]:
# Generate ~5,000 telemetry readings over the past 90 days
NUM_READINGS = 5000
base_time = datetime(2025, 1, 1, 0, 0, 0)
vins = [v.vin for v in vehicles]

# Some vehicles have degrading health (for predictive maintenance signals)
degrading_vins = set(random.sample(vins, 15))  # 15% of fleet

readings = []
for _ in range(NUM_READINGS):
    vin = random.choice(vins)
    ts = base_time + timedelta(
        days=random.randint(0, 90),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    is_degrading = vin in degrading_vins
    
    # Normal ranges with anomalies for degrading vehicles
    engine_temp = random.gauss(195, 10) if not is_degrading else random.gauss(230, 20)
    oil_pressure = random.gauss(40, 5) if not is_degrading else random.gauss(25, 8)
    tire_pressure_fl = random.gauss(32, 1.5)
    tire_pressure_fr = random.gauss(32, 1.5)
    tire_pressure_rl = random.gauss(32, 1.5)
    tire_pressure_rr = random.gauss(32, 1.5)
    battery_voltage = random.gauss(12.6, 0.3) if not is_degrading else random.gauss(11.5, 0.8)
    rpm = random.gauss(2500, 800) if not is_degrading else random.gauss(3500, 1200)
    speed_mph = max(0.0, random.gauss(45, 25))
    fuel_level_pct = random.uniform(5, 100)
    
    readings.append(Row(
        reading_id=str(uuid.uuid4()),
        vin=vin,
        timestamp=ts,
        engine_temp_f=float(round(engine_temp, 1)),
        oil_pressure_psi=float(round(max(0.0, oil_pressure), 1)),
        tire_pressure_fl_psi=float(round(tire_pressure_fl, 1)),
        tire_pressure_fr_psi=float(round(tire_pressure_fr, 1)),
        tire_pressure_rl_psi=float(round(tire_pressure_rl, 1)),
        tire_pressure_rr_psi=float(round(tire_pressure_rr, 1)),
        battery_voltage_v=float(round(battery_voltage, 2)),
        rpm=int(max(0, rpm)),
        speed_mph=float(round(speed_mph, 1)),
        fuel_level_pct=float(round(fuel_level_pct, 1))
    ))

df_telemetry = spark.createDataFrame(readings)
df_telemetry = df_telemetry.withColumn("_ingested_at", current_timestamp()) \
                           .withColumn("_source", lit("synthetic_generator_v1"))

df_telemetry.write.format("delta").mode("overwrite").save("Tables/raw_telemetry")
print(f"Wrote {df_telemetry.count()} telemetry readings to Bronze raw_telemetry")

## Generate Diagnostic Trouble Codes (DTCs)

In [ ]:
# DTC codes relevant to predictive maintenance
DTC_CODES = [
    ("P0115", "Engine Coolant Temperature Circuit", "Engine", "High"),
    ("P0217", "Engine Overtemperature Condition", "Engine", "Critical"),
    ("P0520", "Engine Oil Pressure Sensor Circuit", "Engine", "High"),
    ("P0524", "Engine Oil Pressure Too Low", "Engine", "Critical"),
    ("P0562", "System Voltage Low", "Electrical", "Medium"),
    ("P0563", "System Voltage High", "Electrical", "Medium"),
    ("P0300", "Random/Multiple Cylinder Misfire", "Engine", "High"),
    ("P0420", "Catalyst System Efficiency Below Threshold", "Emissions", "Medium"),
    ("P0171", "System Too Lean (Bank 1)", "Fuel", "Medium"),
    ("P0174", "System Too Lean (Bank 2)", "Fuel", "Medium"),
    ("P0442", "EVAP System Leak Detected (Small)", "Emissions", "Low"),
    ("P0455", "EVAP System Leak Detected (Large)", "Emissions", "Medium"),
    ("P0128", "Coolant Thermostat Below Regulating Temperature", "Engine", "Low"),
    ("P0507", "Idle Air Control System RPM Higher Than Expected", "Engine", "Low"),
    ("P0700", "Transmission Control System Malfunction", "Transmission", "High")
]

NUM_DTCS = 3000
dtc_records = []
for _ in range(NUM_DTCS):
    vin = random.choice(vins)
    is_degrading = vin in degrading_vins
    
    # Degrading vehicles get more critical DTCs
    if is_degrading:
        code, desc, system, severity = random.choice(
            [d for d in DTC_CODES if d[3] in ("High", "Critical")]
        )
    else:
        code, desc, system, severity = random.choice(DTC_CODES)
    
    ts = base_time + timedelta(
        days=random.randint(0, 90),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    
    dtc_records.append(Row(
        dtc_event_id=str(uuid.uuid4()),
        vin=vin,
        timestamp=ts,
        dtc_code=code,
        dtc_description=desc,
        system_category=system,
        severity=severity,
        is_active=random.choice([True, True, True, False])  # 75% active
    ))

df_dtcs = spark.createDataFrame(dtc_records)
df_dtcs = df_dtcs.withColumn("_ingested_at", current_timestamp()) \
                  .withColumn("_source", lit("synthetic_generator_v1"))

df_dtcs.write.format("delta").mode("overwrite").save("Tables/raw_dtc_events")
print(f"Wrote {df_dtcs.count()} DTC events to Bronze raw_dtc_events")

## Generate Maintenance Records

In [ ]:
MAINTENANCE_TYPES = [
    ("Oil Change", "Routine", 75.0),
    ("Tire Rotation", "Routine", 50.0),
    ("Brake Pad Replacement", "Repair", 350.0),
    ("Battery Replacement", "Repair", 200.0),
    ("Engine Diagnostic", "Diagnostic", 120.0),
    ("Transmission Service", "Repair", 500.0),
    ("Coolant Flush", "Routine", 100.0),
    ("Air Filter Replacement", "Routine", 30.0),
    ("Spark Plug Replacement", "Repair", 180.0),
    ("Wheel Alignment", "Routine", 90.0),
    ("Engine Overhaul", "Major Repair", 3500.0),
    ("Turbo Replacement", "Major Repair", 2500.0)
]

NUM_MAINTENANCE = 2000
maint_records = []
for _ in range(NUM_MAINTENANCE):
    vin = random.choice(vins)
    mtype, category, base_cost = random.choice(MAINTENANCE_TYPES)
    cost = round(base_cost * random.uniform(0.8, 1.4), 2)
    ts = base_time + timedelta(
        days=random.randint(0, 90),
        hours=random.randint(8, 17)
    )
    
    maint_records.append(Row(
        maintenance_id=str(uuid.uuid4()),
        vin=vin,
        service_date=ts,
        maintenance_type=mtype,
        category=category,
        cost_usd=cost,
        dealer_id=f"DLR-{random.randint(1, 50):04d}",
        technician_id=f"TECH-{random.randint(1, 200):04d}",
        parts_replaced=random.randint(0, 5),
        labor_hours=round(random.uniform(0.5, 8.0), 1),
        warranty_claim=random.choice([True, False, False, False])  # 25% warranty
    ))

df_maintenance = spark.createDataFrame(maint_records)
df_maintenance = df_maintenance.withColumn("_ingested_at", current_timestamp()) \
                                .withColumn("_source", lit("synthetic_generator_v1"))

df_maintenance.write.format("delta").mode("overwrite").save("Tables/raw_maintenance")
print(f"Wrote {df_maintenance.count()} maintenance records to Bronze raw_maintenance")

## Summary

In [ ]:
print("=== Bronze Layer Ingestion Complete ===")
print(f"  raw_vehicles:    {spark.read.format('delta').load('Tables/raw_vehicles').count()} rows")
print(f"  raw_telemetry:   {spark.read.format('delta').load('Tables/raw_telemetry').count()} rows")
print(f"  raw_dtc_events:  {spark.read.format('delta').load('Tables/raw_dtc_events').count()} rows")
print(f"  raw_maintenance: {spark.read.format('delta').load('Tables/raw_maintenance').count()} rows")